In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#Skeletonization

import os
import numpy as np
from skimage.io import imread, imsave
from skimage.morphology import skeletonize

def skeletonize_folder(input_folder, output_folder):
#Applies skeletonization to all binary images in a folder.
#Saves results in output_folder with the same filenames.

    os.makedirs(output_folder, exist_ok=True)
    files = [f for f in os.listdir(input_folder) if not f.startswith('.')]
    print(f"Found {len(files)} images.")
    for fname in files:
        in_path = os.path.join(input_folder, fname)
        out_path = os.path.join(output_folder, fname)

        # Read image
        img = imread(in_path, as_gray=True)

        # Skeletonization
        skel = skeletonize(img)

        # Convert back to uint8 for saving
        skel_uint8 = (skel * 255).astype(np.uint8)

        imsave(out_path, skel_uint8)

    print(f"Skeletons saved in: {output_folder}")


# CONFIGURATION AND RUN
if __name__ == "__main__":
    input_folder = "/content/drive/MyDrive/..." #folder that contains pre-processed masks
    output_folder = "/content/drive/MyDrive/..." #folder that contains skeletonized masks

    if os.path.exists(input_folder):
        skeletonize_folder(input_folder, output_folder)
    else:
        print("Error: input folder does not exist.")


In [ ]:
## Compute clDice
import os
import numpy as np
import matplotlib.pyplot as plt
from skimage.io import imread
from skimage.morphology import skeletonize

def compute_cldice(y_true, y_pred):
#Computes the clDice score (Centerline Dice) between Ground Truth and Prediction.

    # 1. Convert inputs to boolean masks (ensure binary)
    t_prec = y_true > 0
    s_prec = y_pred > 0

    # Handle empty image cases
    if np.sum(t_prec) == 0 and np.sum(s_prec) == 0:
        return 1.0
    if np.sum(t_prec) == 0 or np.sum(s_prec) == 0:
        return 0.0

    # 2. Skeletonization
    t_skel = skeletonize(t_prec)  # Ground Truth skeleton (Sx)
    s_skel = skeletonize(s_prec)  # Prediction skeleton (Sy)

    # 3. Topology Precision (TP): how much of predicted skeleton is correct?
    # Formula: |Sy ∩ X| / |Sy|
    tp_num = np.sum(s_skel & t_prec)
    tp_den = np.sum(s_skel)
    t_prec_score = tp_num / tp_den if tp_den > 0 else 0

    # 4. Topology Sensitivity (TS): how much of GT skeleton was found?
    # Formula: |Sx ∩ Y| / |Sx|
    ts_num = np.sum(t_skel & s_prec)
    ts_den = np.sum(t_skel)
    t_sens_score = ts_num / ts_den if ts_den > 0 else 0

    # 5. Harmonic mean (clDice)
    if (t_prec_score + t_sens_score) == 0:
        return 0.0

    cl_dice = 2 * (t_prec_score * t_sens_score) / (t_prec_score + t_sens_score)
    return cl_dice

def evaluate_dataset(manual_folder, auto_folder):
#Iterates over the dataset and computes clDice using the custom filename matching logic.

    # Get file lists (ignore hidden files like .DS_Store)
    manual_files = [f for f in os.listdir(manual_folder) if not f.startswith('.')]
    auto_files = [f for f in os.listdir(auto_folder) if not f.startswith('.')]

    scores = []
    found_count = 0

    print(f"Start evaluation. Manual: {len(manual_files)}, Automatic: {len(auto_files)}")

    for manual_fname_str in manual_files:
        # Extract base name without extension
        base_name = os.path.splitext(manual_fname_str)[0]

        # --- CUSTOM MATCHING LOGIC ---
        matched_auto = None
        for auto_fname in auto_files:
            if base_name in auto_fname:
                matched_auto = auto_fname
                break  # Stop at first match

        if matched_auto is None:
            print(f"No automatic mask found for: {manual_fname_str}")
            continue

        # Load paired images
        try:
            gt_path = os.path.join(manual_folder, manual_fname_str)
            pred_path = os.path.join(auto_folder, matched_auto)

            # Read as grayscale
            gt_img = imread(gt_path, as_gray=True)
            pred_img = imread(pred_path, as_gray=True)

            # Compute clDice
            val = compute_cldice(gt_img, pred_img)
            scores.append(val)
            found_count += 1

        except Exception as e:
            print(f"Error reading {manual_fname_str}: {e}")

    # Final statistics
    if not scores:
        print("No scores computed.")
        return

    scores_arr = np.array(scores)
    mean_val = np.mean(scores_arr)
    std_val = np.std(scores_arr)

    # Text output
    print(f"Images processed: {found_count}/{len(manual_files)}")
    print(f"clDice Result: {mean_val:.4f} ± {std_val:.4f}")

    # Graphical output (boxplot)
    plt.figure(figsize=(4, 6))
    plt.boxplot(scores_arr)
    plt.title("clDice Distribution")
    plt.ylabel("Score")
    # Plot mean point
    plt.plot(1, mean_val, 'r.', markersize=10, label='Mean')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

# --- CONFIGURATION AND RUN ---
if __name__ == "__main__":
    # Insert your folder paths here
    manual_folder = "/content/drive/MyDrive/..." #path of the manual (post-processed) masks
    auto_folder = "/content/drive/MyDrive/..." #path of the automatic masks

    if os.path.exists(manual_folder) and os.path.exists(auto_folder):
        evaluate_dataset(manual_folder, auto_folder)
    else:
        print("Error: please provide valid folder paths in the __main__ block")
